# B2.8 · Exploit chaining

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.7 · Dynamic exploitation (DAST)](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**.

| | |
|---|---|
| Tools used | OWASP ZAP, Kimi K2, Claude Opus 5 |

## What this lesson is

**What it covers.** Chain individually-medium findings into a critical path and show the severity the chain earns.

**Why a security engineer needs it.** Three medium findings are triaged as three mediums, and nobody notices they compose. The control it builds is: stage 13: combine validated findings into multi-step sequences and score the chain, not the links.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Three medium findings, each correctly scored, each individually not worth an engineer's afternoon. Chained, they read a file that ends the conversation about severity. Chains are where automated analysis earns its keep.

> **At CyberTravels.** Verbose errors, an open redirect and a path traversal are each low on their own. Chained against CyberTravels they read a config file and end the conversation about severity. R9.

## 2 · The framework

```
   alone                                chained
   +----------------+                   +-------------------------+
   | path traversal | medium            | traversal reads config  |
   | verbose errors | low        --->   | errors leak the key     |
   | open redirect  | low               | redirect delivers it    |
   +----------------+                   +-------------------------+
                                        outcome: credential exfiltration

   severity is a property of the chain, not of the link
```

**Stage 13 — Exploit chaining.** Individual findings are triaged individually,
and that is how three mediums become a critical nobody noticed.

The arithmetic of severity is not additive. A read-only information disclosure
is a medium. A CSRF is a medium. An unauthenticated internal endpoint is a
medium. Chained — leak an ID, forge a request using it, hit the internal
endpoint with the forged session — the outcome is account takeover, which is
not a medium.

The pipeline can find these mechanically because Phase 4 already produced
confirmed findings with known **preconditions** and **effects**. If one
finding's effect satisfies another's precondition, they compose, and the chain's
severity is the severity of its final effect.

This is the stage that most often changes what gets fixed first.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Phase 4 as a skill — and the preconditions that gate it

Dynamic validation is the one phase that *acts*. Everything before it reads; this one sends input to a running system. The skill therefore opens with safety preconditions rather than a procedure, and a refusal is a first-class output.

The contract also insists that `reproduced: false` be reported rather than dropped. A finding that survived Phase 3 and then failed to reproduce is the most useful signal the pipeline produces about its own false-positive rate — and it is the one a tidy report deletes.

### The skill — [`skills/appsec/appsec-exploit-validate/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-exploit-validate/SKILL.md)

```yaml
name: appsec-exploit-validate
description: >-
  Prove a candidate vulnerability is real by reproducing it in an isolated
  sandbox, then decide whether it chains into something worse. Use when asked
  to validate or confirm a finding, write or run a proof-of-concept, check
  exploitability, reproduce a CVE, or separate theoretical findings from
  demonstrated ones.
allowed-tools: Bash, Read, Write, Grep
```

# AppSec pipeline · Phase 4 — Dynamic validation

Covers **stages 11–14**. A finding that has been reproduced is a different
object from a finding that has been argued for. This phase produces the
difference, and it is the phase with the sharpest safety constraints.

## When to use this

Only on findings that reached `feasible: true, verdict: confirmed` in Phase 3.
Validating an unfiltered list wastes the most expensive stage in the pipeline
on findings Phase 3 would have deleted.

## Safety preconditions — check before anything else

Refuse to proceed unless **all** hold, and say which one failed:

1. The target is a **local sandbox or an explicitly authorised environment**.
   Never a production host, never a third party's infrastructure.
2. The sandbox has **no credentials** beyond throwaway ones, and **no route**
   to a real network the exploit could reach.
3. The proof-of-concept **demonstrates** the flaw. It does not exfiltrate real
   data, persist, escalate beyond the demonstration, or damage anything.
4. There is a **teardown** and it runs even on failure.

This is not ceremony. A validation harness with production credentials in its
environment is itself the vulnerability.

## Procedure

**Stage 11 — Sandbox replication.** Stand the component up in isolation with
the smallest fixture that reproduces the conditions: the vulnerable version,
the reachable entry point, and nothing else. Record the exact commit and the
fixture, because a PoC that cannot be re-run is an anecdote.

**Stage 12 — Dynamic exploitation.** Drive the entry point with an input that
should trigger the sink. Capture the observable: the crash, the query executed,
the file read, the process spawned. **The observable is the evidence** — an
exit code is not. Record what you sent and what came back.

**Stage 13 — Exploit chaining.** Ask what the demonstrated primitive gives
access to next. A path traversal that reads a config file containing a token is
not a file-read bug; it is a credential-disclosure bug. Chain only within the
sandbox, and stop at the first step that would need a real credential.

**Stage 14 — Remediation engineering.** Propose the fix at the right layer, and
say what it costs. Prefer the control that makes the class impossible
(parameterised queries, an allowlist, a type) over the one that blocks the
sample payload (a regex on the input you happened to send). Then **re-run
stage 12 against the fix** — a remediation that has not been tested against the
PoC that motivated it is a hypothesis.

## Output contract

```json
{
  "validations": [
    {"finding_id": "str", "reproduced": true,
     "sandbox": {"commit": "str", "fixture": "str", "isolated": true},
     "input": "str", "observable": "str",
     "chain": [{"primitive": "str", "leads_to": "str", "stopped_because": "str"}],
     "remediation": {"layer": "input|query|api|config|architecture",
                     "change": "str", "cost": "low|medium|high",
                     "retested": true, "still_reproduces": false}}
  ],
  "refused": [{"finding_id": "str", "precondition_failed": 1, "why": "str"}]
}
```

`reproduced: false` is a first-class result and must be reported, not dropped.
A finding that survived Phase 3 and then failed to reproduce is the single most
useful signal the pipeline produces about its own false-positive rate.

## Failure modes

- **Reporting `reproduced: true` without an observable.** The observable is the
  claim; without it there is nothing to check.
- **Treating "the exploit ran" as "the fix works".** Re-run after remediation
  or set `retested: false` and say so.
- **Chaining outside the sandbox.** Stop and record `stopped_because`.
- **Fixing the payload instead of the class.** A regex that blocks `../` does
  not fix path traversal.

## Handoff

Pass `validations` to **appsec-triage-report**. Findings that failed to
reproduce keep their finding record and gain `reproduced: false` — the report
must be able to distinguish "demonstrated" from "asserted".

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-exploit-validate/scripts/appsec_exploit_validate.py
SCRIPT = "skills/appsec/appsec-exploit-validate/scripts/appsec_exploit_validate.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · Where it breaks — the tidy report

Now suppose two of these findings do not reproduce, and the pipeline does the natural thing with them.

## What you just proved

Six confirmed findings compose into multiple chains. The highest individual severity is high while the highest chained severity is critical, and at least one critical chain is built entirely from medium-or-lower links — for example SSRF granting internal network access, then the unauthenticated admin endpoint. Remediation ordering puts a medium finding first because it breaks the most chains.

## Your turn

Take your current open findings and write down each one's preconditions and effects. The chaining falls out mechanically, and the finding you should fix first is usually not the one at the top of the severity-sorted queue.

---

**Next → [B2.9 · Remediation engineering](https://spbreed.github.io/cyber-commons/lessons/B2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*